# OCR Ingest & RAG with Visual Grounding

This notebook covers two phases:

| Phase | What happens |
|---|---|
| **1. Ingest** | Parse a PDF with Landing AI ADE → embed chunks → store in Postgres with bounding-box metadata |
| **2. RAG + Visual Grounding** | Query the vector store → retrieve matching chunks → highlight their bounding boxes on the original PDF page |

The bbox columns (`bbox_left`, `bbox_top`, `bbox_right`, `bbox_bottom`) are stored in Postgres as normalised coordinates (0–1 range) so they can be applied to any render resolution.

## 1. Imports & Configuration

In [63]:
import os
import sys
from pathlib import Path

import pymupdf
from dotenv import load_dotenv
from IPython.display import Image as DisplayImage
from IPython.display import Markdown, display
from PIL import Image as PILImage
from PIL import ImageDraw

# Make src/ importable from the notebook
PROJECT_ROOT = Path("../").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(PROJECT_ROOT / ".env", override=True)

# Project imports (after sys.path is set)
from config import EMBEDDING_MODEL, PG_TABLE
from helper import CHUNK_TYPE_COLORS, extract_chunk_image, print_document
from ocr_extraction import DocumentAI
from vector_store import get_retriever, get_store

print("Imports OK")

Imports OK


In [64]:
# ── Configure these paths before running ────────────────────────────────────
CASE_NUMBER   = "SteveGoodman"   # must match the sub-directory under data/
DOCUMENT_PATH = PROJECT_ROOT / f"data/{CASE_NUMBER}/5573DraftAccounts.pdf"
# ─────────────────────────────────────────────────────────────────────────────

assert DOCUMENT_PATH.exists(), f"Document not found: {DOCUMENT_PATH}"
print(f"Document : {DOCUMENT_PATH}")
print(f"Case     : {CASE_NUMBER}")

Document : /Users/stevegoodman/dev/fionaa-be/data/SteveGoodman/5573DraftAccounts.pdf
Case     : SteveGoodman


---
## 2. Phase 1 — Ingest

### 2a. Parse with Landing AI ADE
Calls `client.parse()` which returns:
- `.splits[]` — per-page content
- `.chunks[]` — individual content blocks (text / table / figure …)
- `.grounding` — `dict[chunk_id, Grounding]` with page + normalised bbox

In [ ]:
document = DocumentAI(DOCUMENT_PATH, case_number=CASE_NUMBER)
document.parse()

parse_result = document.parse_result
print(f"Pages  : {len(parse_result.splits)}")
print(f"Chunks : {len(parse_result.chunks)}")
print(f"Grounding entries: {len(parse_result.grounding)}")

### 2b. Inspect a grounding entry
Each entry in `parse_result.grounding` is keyed by `chunk_id` and has:
- `.page` — 0-indexed page number
- `.type` — chunk type string (`chunkText`, `chunkTable`, …)
- `.box.left / .top / .right / .bottom` — normalised (0–1) coordinates

In [ ]:
sample_id, sample_g = next(iter(parse_result.grounding.items()))
print(f"chunk_id : {sample_id}")
print(f"page     : {sample_g.page}")
print(f"type     : {sample_g.type}")
print(f"box      : left={sample_g.box.left:.4f}  top={sample_g.box.top:.4f}  "
      f"right={sample_g.box.right:.4f}  bottom={sample_g.box.bottom:.4f}")

### 2c. Visualise all grounded chunks (optional sanity check)

In [ ]:
TARGET_PAGE = 0   # change to see other pages

pdf = pymupdf.open(DOCUMENT_PATH)
page = pdf[TARGET_PAGE]
pix  = page.get_pixmap(matrix=pymupdf.Matrix(2, 2))
img  = PILImage.frombytes("RGB", [pix.width, pix.height], pix.samples)
pdf.close()

draw = ImageDraw.Draw(img)
img_w, img_h = img.size
for cid, g in parse_result.grounding.items():
    if g.page != TARGET_PAGE:
        continue
    b = g.box
    color = CHUNK_TYPE_COLORS.get(g.type, (128, 128, 128))
    x1, y1 = int(b.left * img_w), int(b.top * img_h)
    x2, y2 = int(b.right * img_w), int(b.bottom * img_h)
    draw.rectangle([x1, y1, x2, y2], outline=color, width=2)

display(img.resize((int(img_w * 0.5), int(img_h * 0.5))))

In [ ]:
# Clear existing rows for this case before re-ingesting
from sqlalchemy import text
from sqlalchemy.ext.asyncio import create_async_engine

engine = create_async_engine(
    f"postgresql+asyncpg://{POSTGRES_USER}:{POSTGRES_PASSWORD}"
    f"@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
)
async with engine.begin() as conn:
    result = await conn.execute(
        text(f'DELETE FROM "{PG_TABLE}" WHERE case_number = :case'),
        {"case": CASE_NUMBER},
    )
    print(f"Deleted {result.rowcount} rows for case '{CASE_NUMBER}'")
await engine.dispose()

### 2d. Embed chunks and store in Postgres

`embed_and_store()` persists **bounding-box coordinates** alongside each chunk so they can be retrieved for visual grounding later.

> **First run only** — run `uv run python src/vector_store.py` from the project root to create / migrate the table schema before embedding.

In [ ]:
stored = await document.embed_and_store()
print(f"Stored {stored} chunks in table '{PG_TABLE}'.")

---
## 3. Phase 2 — RAG with Visual Grounding

### 3a. Connect to the vector store

In [ ]:
store = await get_store()
print("Connected to PGVectorStore")

### 3b. Visual-grounding query function

For each retrieved chunk the function:
1. Renders the source PDF page at 2× resolution
2. Crops to the stored bounding box (with padding)
3. Displays the cropped image alongside the text excerpt

In [ ]:
async def rag_query(
    question: str,
    k: int = 5,
    chunk_type: str | None = None,
    show_images: bool = True,
):
    """Semantic search against the Postgres vector store with visual grounding.

    Args:
        question:    Natural-language query.
        k:           Max chunks to retrieve.
        chunk_type:  Optional filter — e.g. 'chunkTable', 'chunkText'.
        show_images: Whether to render cropped bounding-box images.

    Returns:
        List of retrieved LangChain Documents.
    """
    retriever = get_retriever(store, CASE_NUMBER, k=k, chunk_type=chunk_type)
    docs = await retriever.ainvoke(question)

    display(Markdown(f"### Query: *{question}*\n---"))
    for i, doc in enumerate(docs, 1):
        meta = doc.metadata
        page_num    = meta.get("page_num") or 0
        ctype       = meta.get("chunk_type") or "unknown"
        chunk_id    = meta.get("chunk_id") or ""
        bbox_left   = meta.get("bbox_left")
        bbox_top    = meta.get("bbox_top")
        bbox_right  = meta.get("bbox_right")
        bbox_bottom = meta.get("bbox_bottom")

        display(Markdown(
            f"**Result {i}** — type: `{ctype}` | page: {page_num} | chunk: `{chunk_id[:8] if chunk_id else 'N/A'}…`\n\n"
            f"```\n{doc.page_content[:300]}\n```"
        ))

        if show_images and all(v is not None for v in [bbox_left, bbox_top, bbox_right, bbox_bottom]):
            bbox = [bbox_left, bbox_top, bbox_right, bbox_bottom]
            img_bytes = extract_chunk_image(
                pdf_path=DOCUMENT_PATH,
                page_num=page_num,
                bbox=bbox,
                highlight=True,
                padding=15,
            )
            if img_bytes:
                display(DisplayImage(data=img_bytes))
            else:
                print("  (could not extract chunk image)")
        elif show_images:
            print(f"  (no bbox stored — re-ingest this document to get visual grounding)")
        print("-" * 60)

    return docs

print("rag_query() defined")

### 3c. Run a query

In [ ]:
results = await rag_query(
    "What is the turnover and profit for the company?",
    k=4,
)

### 3d. Hybrid search — tables only

Narrow retrieval to `chunkTable` chunks to focus on financial figures.

In [ ]:
results_tables = await rag_query(
    "balance sheet fixed assets and debtors",
    k=3,
    chunk_type="chunkTable",
)

---
## 4. Full RAG Chain with Claude

Combine the PGVectorStore retriever with a LangChain RAG chain backed by Claude.

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)

system_prompt = (
    "You are a financial analyst assistant. "
    "Use ONLY the retrieved document chunks below to answer. "
    "If the answer is not in the context, say so.\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

retriever = get_retriever(store, CASE_NUMBER, k=6)
doc_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, doc_chain)

print("RAG chain ready")

In [ ]:
response = await rag_chain.ainvoke({"input": "What are the company's total turnover and net profit for the most recent year?"})
display(Markdown(f"**A:** {response['answer']}"))

### 4a. Ground the answer — show source chunks

In [ ]:
display(Markdown("**Source chunks used to generate the answer:**"))
for doc in response["context"]:
    meta = doc.metadata
    page_num    = meta.get("page_num", 0)
    ctype       = meta.get("chunk_type", "unknown")
    bbox_left   = meta.get("bbox_left")
    bbox_top    = meta.get("bbox_top")
    bbox_right  = meta.get("bbox_right")
    bbox_bottom = meta.get("bbox_bottom")

    display(Markdown(f"*page {page_num} | {ctype}*\n```\n{doc.page_content[:200]}\n```"))

    if all(v is not None for v in [bbox_left, bbox_top, bbox_right, bbox_bottom]):
        img_bytes = extract_chunk_image(
            pdf_path=DOCUMENT_PATH,
            page_num=page_num,
            bbox=[bbox_left, bbox_top, bbox_right, bbox_bottom],
            highlight=True,
            padding=15,
        )
        if img_bytes:
            display(DisplayImage(data=img_bytes))

---
## 5. Inspect the LangGraph Cloud Store

Connect directly to the LangGraph Platform managed store to verify what chunks
have been indexed and test semantic search against the live deployment.

**Requires** `LANGGRAPH_URL` in `.env` — the deployment URL from LangGraph Cloud
(e.g. `https://fionaa-abc123.us.langgraph.app`).

In [92]:
### 5a. Connect to the LangGraph Cloud store client
from langgraph_sdk import get_client

LANGGRAPH_URL = "https://fionaa-cloud-767cb01f2a6c5b8983d07b65f27fc4da.us.langgraph.app".strip()
LANGSMITH_API_KEY = os.environ.get("LANGSMITH_API_KEY", "").strip()

assert LANGGRAPH_URL, "Set LANGGRAPH_URL in .env (e.g. https://fionaa-xxx.us.langgraph.app)"

lg_client = get_client(url=LANGGRAPH_URL, api_key=LANGSMITH_API_KEY)
print(f"Connected to {LANGGRAPH_URL}")

Connected to https://fionaa-cloud-767cb01f2a6c5b8983d07b65f27fc4da.us.langgraph.app


In [78]:
### 5b. List ALL namespaces in the store (no filter — shows everything)
ns_response = await lg_client.store.list_namespaces(limit=100)
all_namespaces = ns_response["namespaces"]

print(f"Total namespaces in store: {len(all_namespaces)}")
for ns in all_namespaces:
    print(f"  {ns}")

if not all_namespaces:
    print("  (store is empty — no data has been indexed yet)")

2026-03-12 11:30:14,688 - INFO - HTTP Request: POST https://fionaa-cloud-767cb01f2a6c5b8983d07b65f27fc4da.us.langgraph.app/store/namespaces "HTTP/1.1 200 OK"


Total namespaces in store: 0
  (store is empty — no data has been indexed yet)


In [80]:
### 5b-ii. Confirm store connectivity with a write/read round-trip
import uuid

test_key = f"notebook-test-{uuid.uuid4().hex[:8]}"
test_ns = ("notebook", "connectivity-check")

# Write
await lg_client.store.put_item(test_ns, test_key, {"hello": "world", "ts": "2026-03-12"})
print(f"Wrote test item: key={test_key}")

# Read back
read_back = await lg_client.store.get_item(test_ns, test_key)
print(f"Read back: {read_back}")

# Clean up
await lg_client.store.delete_item(test_ns, test_key)
print("Deleted test item — store connectivity OK")

2026-03-12 11:37:49,376 - INFO - HTTP Request: PUT https://fionaa-cloud-767cb01f2a6c5b8983d07b65f27fc4da.us.langgraph.app/store/items "HTTP/1.1 204 No Content"
2026-03-12 11:37:49,503 - INFO - HTTP Request: GET https://fionaa-cloud-767cb01f2a6c5b8983d07b65f27fc4da.us.langgraph.app/store/items?namespace=notebook.connectivity-check&key=notebook-test-3fd94a0a "HTTP/1.1 200 OK"


Wrote test item: key=notebook-test-3fd94a0a
Read back: {'namespace': ['notebook', 'connectivity-check'], 'key': 'notebook-test-3fd94a0a', 'value': {'ts': '2026-03-12', 'hello': 'world'}, 'created_at': '2026-03-12T11:37:49.318098+00:00', 'updated_at': '2026-03-12T11:37:49.318098+00:00'}


2026-03-12 11:37:49,638 - INFO - HTTP Request: DELETE https://fionaa-cloud-767cb01f2a6c5b8983d07b65f27fc4da.us.langgraph.app/store/items "HTTP/1.1 204 No Content"


Deleted test item — store connectivity OK


In [ ]:
### 5b-iii. List recent threads and their runs to check if startup_node has executed
threads = await lg_client.threads.search(limit=10)
print(f"Recent threads: {len(threads)}\n")

for t in threads:
    tid = t.get("thread_id") or t.get("id")
    created = t.get("created_at")
    metadata = t.get("metadata", {})
    print(f"  thread={tid}  created={created}  metadata={metadata}")

    # List runs on this thread
    runs = await lg_client.runs.list(tid, limit=5)
    for r in runs:
        print(f"    run={r.get('run_id')}  status={r.get('status')}  graph={r.get('assistant_id')}  created={r.get('created_at')}")
    print()

### 5b-iv. Trigger the fionaa graph to ingest documents into the store

The `fionaa` graph has never run on this deployment — `startup_node` (which calls
`embed_and_store`) must execute at least once to populate the store.

Run the cell below to kick off ingestion. It creates a new thread and waits for
the run to complete. **Requires** the source documents to already be in GCS under
`<CASE_NUMBER>/loan_application/`.

In [87]:
### Trigger fionaa graph — runs startup_node only (stops before assessment_deepagent)
# interrupt_before stops execution after startup_node completes, so we get
# OCR + embed_and_store without running the expensive deep agent.

thread = await lg_client.threads.create()
thread_id = thread["thread_id"]
print(f"Created thread: {thread_id}")

result = await lg_client.runs.wait(
    thread_id,
    assistant_id="fionaa",
    input={"case_number": CASE_NUMBER},
    config={"configurable": {"case_number": CASE_NUMBER}},
    interrupt_before=["assessment_deepagent"],
)

print(f"\nstartup_node complete — documents ingested into store.")
docs = (result or {}).get("documents", [])
for d in docs:
    print(f"  {d.get('document_name')}  ({d.get('document_type')})")

2026-03-12 11:55:42,308 - INFO - HTTP Request: POST https://fionaa-cloud-767cb01f2a6c5b8983d07b65f27fc4da.us.langgraph.app/threads "HTTP/1.1 200 OK"
2026-03-12 11:55:42,512 - INFO - HTTP Request: POST https://fionaa-cloud-767cb01f2a6c5b8983d07b65f27fc4da.us.langgraph.app/threads/ab150215-4b8f-4c89-af58-50b1bb136b23/runs/wait "HTTP/1.1 200 OK"


Created thread: ab150215-4b8f-4c89-af58-50b1bb136b23

startup_node complete — documents ingested into store.
  5573DraftAccounts.pdf  (annual_company_report)
  bs_052015.jpeg  (bank_statement)
  bs_102015.jpeg  (bank_statement)


In [96]:
### 5b-v. Inspect the thread state — check what startup_node actually returned
state = await lg_client.threads.get_state(thread_id)
values = state.get("values", {}) if isinstance(state, dict) else state.values

case_num = values.get("case_number")
documents = values.get("documents", [])

print(f"case_number in state : {case_num}")
print(f"documents returned   : {len(documents)}")
for d in documents:
    print(f"  {d.get('document_name')}  type={d.get('document_type')}  ocr_path={d.get('ocr_output_path')}")

if not documents:
    print("\n  ⚠ No documents — startup_node found no source files.")
    print("  Check that files exist in GCS at:")
    print(f"    gs://<BUCKET_NAME>/{CASE_NUMBER}/loan_application/")

2026-03-12 12:06:59,464 - INFO - HTTP Request: GET https://fionaa-cloud-767cb01f2a6c5b8983d07b65f27fc4da.us.langgraph.app/threads/ab150215-4b8f-4c89-af58-50b1bb136b23/state?subgraphs=false "HTTP/1.1 200 OK"


case_number in state : SteveGoodman
documents returned   : 3
  5573DraftAccounts.pdf  type=annual_company_report  ocr_path=/disk-files/SteveGoodman/ocr_output/5573DraftAccounts_extraction.json
  bs_052015.jpeg  type=bank_statement  ocr_path=/disk-files/SteveGoodman/ocr_output/bs_052015_extraction.json
  bs_102015.jpeg  type=bank_statement  ocr_path=/disk-files/SteveGoodman/ocr_output/bs_102015_extraction.json


In [95]:
documents

NameError: name 'documents' is not defined

In [ ]:
### 5b-vi. Manually write a test chunk into the cases namespace and verify it's searchable
test_chunk_key = "manual-test-chunk"

await lg_client.store.put_item(
    ("cases", CASE_NUMBER),
    test_chunk_key,
    {
        "text": "This is a test chunk to verify the store is writable from the notebook.",
        "document_name": "test.pdf",
        "document_type": "test",
        "page_num": 0,
        "chunk_type": "chunkText",
        "bbox_left": 0.1, "bbox_top": 0.2, "bbox_right": 0.9, "bbox_bottom": 0.3,
    },
)
print(f"Wrote test chunk to ('cases', '{CASE_NUMBER}')")

# Read it back
check = await lg_client.store.search_items(("cases", CASE_NUMBER), limit=5)
print(f"Items now in namespace: {len(check['items'])}")
for it in check["items"]:
    print(f"  key={it.get('key')}  doc={it.get('value',{}).get('document_name')}")

2026-03-12 12:04:42,815 - INFO - HTTP Request: PUT https://fionaa-cloud-767cb01f2a6c5b8983d07b65f27fc4da.us.langgraph.app/store/items "HTTP/1.1 204 No Content"
2026-03-12 12:04:42,945 - INFO - HTTP Request: POST https://fionaa-cloud-767cb01f2a6c5b8983d07b65f27fc4da.us.langgraph.app/store/items/search "HTTP/1.1 200 OK"


Wrote test chunk to ('cases', 'SteveGoodman')
Items now in namespace: 1
  key=manual-test-chunk  doc=test.pdf


In [89]:
docs

[{'document_name': '5573DraftAccounts.pdf',
  'document_type': 'annual_company_report',
  'ocr_output_path': '/disk-files/SteveGoodman/ocr_output/5573DraftAccounts_extraction.json'},
 {'document_name': 'bs_052015.jpeg',
  'document_type': 'bank_statement',
  'ocr_output_path': '/disk-files/SteveGoodman/ocr_output/bs_052015_extraction.json'},
 {'document_name': 'bs_102015.jpeg',
  'document_type': 'bank_statement',
  'ocr_output_path': '/disk-files/SteveGoodman/ocr_output/bs_102015_extraction.json'}]

In [90]:
CASE_NUMBER

'SteveGoodman'

In [91]:
### 5c. List items stored for this case — verify chunk count and bbox presence
# search_items with no query returns all items up to limit
response = await lg_client.store.search_items(("cases", CASE_NUMBER), limit=100)
items = response["items"]

print(f"Total chunks stored for case '{CASE_NUMBER}': {len(items)}\n")

has_bbox = sum(1 for it in items if it.get("value", {}).get("bbox_left") is not None)
print(f"Chunks with bbox coordinates: {has_bbox}/{len(items)}\n")

print(f"{'KEY':36}  {'DOC':30}  {'PAGE':4}  {'TYPE':14}  {'BBOX?'}")
print("-" * 100)
for it in items[:20]:
    v = it.get("value", {})
    bbox_flag = "yes" if v.get("bbox_left") is not None else "no "
    print(
        f"{str(it.get('key',''))[:36]:36}  "
        f"{str(v.get('document_name',''))[:30]:30}  "
        f"{str(v.get('page_num','')):4}  "
        f"{str(v.get('chunk_type',''))[:14]:14}  "
        f"{bbox_flag}"
    )
if len(items) > 20:
    print(f"  ... and {len(items) - 20} more")

2026-03-12 12:01:56,546 - INFO - HTTP Request: POST https://fionaa-cloud-767cb01f2a6c5b8983d07b65f27fc4da.us.langgraph.app/store/items/search "HTTP/1.1 200 OK"


Total chunks stored for case 'SteveGoodman': 0

Chunks with bbox coordinates: 0/0

KEY                                   DOC                             PAGE  TYPE            BBOX?
----------------------------------------------------------------------------------------------------


In [76]:
for item in items:
    print(_)

In [77]:
### 5d. Semantic search against the live store
SEARCH_QUERY = "total turnover and net profit"   # ← change this

results = await lg_client.store.search_items(
    ("cases", CASE_NUMBER),
    query=SEARCH_QUERY,
    limit=5,
)

items = results["items"]
print(f"Query: '{SEARCH_QUERY}'  →  {len(items)} result(s)\n")
for i, it in enumerate(items, 1):
    v = it.get("value", {})
    score = f"{it['score']:.3f}" if it.get("score") is not None else "n/a"
    print(f"[{i}] score={score}  doc={v.get('document_name')}  page={v.get('page_num')}  type={v.get('chunk_type')}")
    if v.get("bbox_left") is not None:
        print(f"     bbox: {v['bbox_left']:.4f},{v['bbox_top']:.4f},{v['bbox_right']:.4f},{v['bbox_bottom']:.4f}")
    else:
        print(f"     bbox: none")
    print(f"     {str(v.get('text',''))[:200]}")
    print()

2026-03-12 11:26:10,994 - INFO - HTTP Request: POST https://fionaa-cloud-767cb01f2a6c5b8983d07b65f27fc4da.us.langgraph.app/store/items/search "HTTP/1.1 200 OK"


Query: 'total turnover and net profit'  →  0 result(s)

